# บทที่ 05 · อ่านผล Backtest อย่างซื่อตรง

เรียนรู้การทบต้น ต้นทุนบนมูลค่าพอร์ต การเปรียบเทียบ benchmark และ drawdown จาก 12 session สังเคราะห์

ก่อนเริ่ม: รู้ผลตอบแทนธรรมดาและ Long/Cash ติดตั้ง numpy==2.5.3 และ pandas==2.3.2 แล้วใช้ Python 3.12 รันจากบนลงล่าง ข้อมูลอยู่ครบในไฟล์ ไม่มี network หรือ brokerage calls

หน่วยเป็น USD สมมติ ไม่มีปฏิทิน/timezone/corporate actions เริ่ม 10,000 USD ถือหุ้นเศษส่วนได้ ไม่ชอร์ต ไม่ leverage เงินสดไม่มีดอกเบี้ย ผลตอบแทนเป็นเปิดถึงปิดของ session และไม่มี overnight return เป้าหมายเป็นแผนที่รู้ก่อนเปิด ต้นทุนคิดจาก NAV ก่อนการเข้า/ออก ไม่ใช่มูลค่าซื้อขายหลังค่าบริการตามบท 4

## 1. ตรวจสภาพแวดล้อม

บันทึกเวอร์ชันเพื่อทำซ้ำ ผลที่แนบมารันด้วย Python 3.12.14, NumPy 2.5.3, pandas 2.3.2

In [1]:
import platform
import numpy as np
import pandas as pd

print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__})


{'python': '3.12.14', 'numpy': '2.5.3', 'pandas': '2.3.2'}


## 2. อ่าน input ก่อนอ่านกำไร

เราตรึงผลตอบแทนและสถานะก่อนเริ่มทดลอง ไม่ได้ใช้ข้อมูลนี้เลือกสัญญาณ แต่ละ weight พร้อมก่อนเปิด session นั้น ห้ามตีความตัวเลขเป็นข้อมูลหุ้นจริง

In [2]:
# Each weight is fixed before its session opens. It is an input, not a fitted rule.
asset_returns = np.array([.01, -.02, .03, -.01, .02, -.04, .01, .03, -.02, .01, .02, -.01])
weights = np.array([0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0])
INITIAL_EQUITY = 10_000.0
print(pd.DataFrame({"session": np.arange(1, 13), "asset_return_pct": asset_returns * 100, "weight": weights}).to_string(index=False))


 session  asset_return_pct  weight
       1               1.0       0
       2              -2.0       1
       3               3.0       1
       4              -1.0       0
       5               2.0       1
       6              -4.0       1
       7               1.0       1
       8               3.0       0
       9              -2.0       0
      10               1.0       1
      11               2.0       1
      12              -1.0       0


## 3. สร้างการบัญชีและ benchmark

ทุกครั้งที่เปลี่ยน Long/Cash หัก c จาก NAV ก่อนซื้อขาย ลงทุนด้วยเงินที่เหลือหลังต้นทุน คิดต้นทุนปิดสถานะท้ายชุดด้วย

V_t = V_(t-1) × (1 - c × |w_t - w_(t-1)|) × (1 + w_t × r_t)

ซื้อแล้วถือใช้ต้นทุนและช่วงเวลาเดียวกัน ดูเงินสดที่จ่ายจริงแยกจากส่วนต่างเงินปลายทางซึ่งรวมผลการทบต้น

In [3]:
def simulate(returns, targets, cost_rate=0.001, initial=INITIAL_EQUITY):
    """Charge a fraction of current NAV at each all-in/cash switch.

    Buy with remaining capital after entry costs; sell the entire holding on exit.
    Liquidate after the last return and include that fee in the last session.
    Targets must be known before each session starts. No shorts or leverage.
    """
    returns, targets = np.asarray(returns, float), np.asarray(targets)
    if len(returns) != len(targets) or len(returns) == 0:
        raise ValueError("Inputs must have equal, nonzero length")
    if not np.isfinite(returns).all() or np.any(returns <= -1):
        raise ValueError("Returns must be finite and greater than -1")
    if not np.isin(targets, [0, 1]).all() or not 0 <= cost_rate < 1 or initial <= 0:
        raise ValueError("Invalid target, cost, or initial capital")
    targets = targets.astype(int)
    equity, previous, total_cost, rows = float(initial), 0, 0.0, []
    for i, (ret, target) in enumerate(zip(returns, targets), start=1):
        before = equity
        fee = equity * cost_rate * abs(target - previous)
        equity = (equity - fee) * (1 + target * ret)
        terminal_fee = equity * cost_rate if i == len(returns) and target else 0.0
        equity -= terminal_fee
        total_cost += fee + terminal_fee
        rows.append((i, target, ret, fee + terminal_fee, equity, equity / before - 1))
        previous = target
    frame = pd.DataFrame(rows, columns=["session", "weight", "asset_return", "cost_usd", "equity_usd", "net_return"])
    frame.attrs["total_cost_usd"] = total_cost
    return frame


def metrics(frame, initial=INITIAL_EQUITY):
    equity = np.r_[initial, frame.equity_usd.to_numpy()]
    peaks = np.maximum.accumulate(equity)
    drawdown = equity / peaks - 1
    return {
        "final_equity_usd": equity[-1],
        "total_return_pct": (equity[-1] / initial - 1) * 100,
        "max_drawdown_pct": drawdown.min() * 100,
        "session_vol_pct": frame.net_return.std(ddof=1) * 100,
        "cash_cost_usd": frame.attrs["total_cost_usd"],
    }


gross = simulate(asset_returns, weights, cost_rate=0)
net = simulate(asset_returns, weights)
benchmark = simulate(asset_returns, np.ones(12, dtype=int))
summary = pd.DataFrame({"strategy_gross": metrics(gross), "strategy_net": metrics(net), "buy_hold_net": metrics(benchmark)}).T
print(summary.round(6).to_string())


                final_equity_usd  total_return_pct  max_drawdown_pct  session_vol_pct  cash_cost_usd
strategy_gross      10284.368382          2.843684              -4.0         1.864745       0.000000
strategy_net        10222.816232          2.228162              -4.0         1.867488      60.286757
buy_hold_net        10255.661177          2.556612              -4.0         2.223075      20.265927


## 4. เปลี่ยนต้นทุน โดยคงกติกา

ทดลอง 0, 5, 10, 20 และ 50 bps ต่อข้าง จำนวนคำสั่งและข้อมูลเดิมคงที่ นี่คือ sensitivity analysis ไม่ใช่การเลือกค่าต้นทุนที่ทำให้ผลดูดี

In [4]:
sensitivity = pd.DataFrame([
    {"cost_bps_per_side": bps, **metrics(simulate(asset_returns, weights, bps / 10_000))}
    for bps in [0, 5, 10, 20, 50]
])
print(sensitivity[["cost_bps_per_side", "final_equity_usd", "total_return_pct", "max_drawdown_pct"]].round(6).to_string(index=False))


 cost_bps_per_side  final_equity_usd  total_return_pct  max_drawdown_pct
                 0      10284.368382          2.843684              -4.0
                 5      10253.553818          2.535538              -4.0
                10      10222.816232          2.228162              -4.0
                20      10161.571381          1.615714              -4.0
                50       9979.668354         -0.203316              -4.0


## 5. วัดทั้งทางลงและทางขึ้น

Drawdown ต้องรวมเงินตั้งต้นใน peak ค่าเป็นลบและรายงานค่าที่ต่ำที่สุด ตัวอย่าง 90% win rate ต่อท้ายใช้กำไรขาดทุนเป็น USD ต่อดีล ไม่ใช่ผลตอบแทนทบต้น

In [5]:
equity = np.r_[INITIAL_EQUITY, net.equity_usd.to_numpy()]
peaks = np.maximum.accumulate(equity)
drawdown = equity / peaks - 1
print(pd.DataFrame({"session": np.arange(13), "equity_usd": equity, "peak_usd": peaks, "drawdown_pct": drawdown * 100}).round(6).to_string(index=False))
trade_pnl = np.array([5.0] * 9 + [-60.0])
print({"winning_trades_pct": float((trade_pnl > 0).mean() * 100), "total_pnl_usd": float(trade_pnl.sum())})


 session   equity_usd     peak_usd  drawdown_pct
       0 10000.000000 10000.000000      0.000000
       1 10000.000000 10000.000000      0.000000
       2  9790.200000 10000.000000     -2.098000
       3 10083.906000 10083.906000      0.000000
       4 10073.822094 10083.906000     -0.100000
       5 10265.023237 10265.023237      0.000000
       6  9854.422308 10265.023237     -4.000000
       7  9952.966531 10265.023237     -3.040000
       8  9943.013564 10265.023237     -3.136960
       9  9943.013564 10265.023237     -3.136960
      10 10032.401256 10265.023237     -2.266161
      11 10233.049281 10265.023237     -0.311484
      12 10222.816232 10265.023237     -0.411173
{'winning_trades_pct': 90.0, 'total_pnl_usd': -15.0}


## 6. ตรวจสมการอย่างอิสระ

เซลล์นี้คูณผลของ 7 session ที่ถือด้วยมือผ่านค่าคงที่ ไม่อ่าน equity จาก loop กลับมาคำนวณคำตอบเดิม ตรวจต้นทุนเข้า/ออกของ benchmark และกรณีที่ช่วงแรกขาดทุนด้วย

In [6]:
# Reconstruct the gross return directly from the seven invested sessions.
expected_gross = INITIAL_EQUITY * .98 * 1.03 * 1.02 * .96 * 1.01 * 1.01 * 1.02
# There are six entry/exit events; each multiplies capital by (1 - cost_rate).
assert int(np.abs(np.diff(np.r_[0, weights, 0])).sum()) == 6
assert np.isclose(gross.equity_usd.iloc[-1], expected_gross)
assert np.isclose(net.equity_usd.iloc[-1], expected_gross * .999**6)
assert np.isclose(benchmark.equity_usd.iloc[-1], INITIAL_EQUITY * np.prod(1 + asset_returns) * .999**2)
# Initial capital must count as a peak even if the first observed return is negative.
tiny = simulate(np.array([-.1, .05]), np.ones(2, dtype=int), cost_rate=0)
assert np.isclose(metrics(tiny)["max_drawdown_pct"], -10)
assert np.all(np.diff(sensitivity.final_equity_usd) < 0)
assert np.isclose(trade_pnl.sum(), -15)
print("PASS: cost compounding, benchmark costs, initial peak, sensitivity, and win-rate counterexample")


PASS: cost compounding, benchmark costs, initial peak, sensitivity, and win-rate counterexample


## แบบฝึกหัด

เปลี่ยน c เป็น 50 bps โดยคง input ทั้งหมด แล้วเทียบกับ buy-and-hold ที่ใช้ c เดียวกัน เขียนคำตอบก่อนรันเซลล์ถัดไป: เงินปลายทางเท่าไร จำนวนการเปลี่ยนสถานะยังเท่าเดิมหรือไม่?

เฉลย: กลยุทธ์ 9,979.67 USD, buy-and-hold 10,173.70 USD, กลยุทธ์เปลี่ยนสถานะ 6 ครั้ง ไม่ควรเทียบ net กับ gross

ข้อสรุปต้องจำกัดอยู่ที่ข้อมูลสังเคราะห์และสมมติฐานนี้ ไม่ annualize ผล 12 session เป็นรายปี ไม่เปลี่ยน parameter หลังดู final test แล้วอ้างว่าเป็น out-of-sample

In [7]:
trial = simulate(asset_returns, weights, cost_rate=.005)
base = simulate(asset_returns, np.ones(12, dtype=int), cost_rate=.005)
print("strategy USD:", round(trial.equity_usd.iloc[-1], 2))
print("benchmark USD:", round(base.equity_usd.iloc[-1], 2))
print("switches:", int(np.abs(np.diff(np.r_[0, weights, 0])).sum()))

strategy USD: 9979.67
benchmark USD: 10173.7
switches: 6


## แหล่งอ้างอิงและสถานะการรัน

Yves Hilpisch, Python for Algorithmic Trading; https://github.com/yhilpisch/py4at; บท 4 หน้า 111–112 (PDF 131–132) และบท 10 หน้า 287–288 (PDF 307–308)

- https://pandas.pydata.org/docs/reference/api/pandas.Series.cummax.html
- https://scikit-learn.org/stable/common_pitfalls.html#data-leakage
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html

ตรวจแหล่งออนไลน์ 11 กันยายน 2026 บทเรียน ข้อมูล และโค้ดเขียนใหม่ ไม่ได้แนบหนังสือหรือแจกโค้ดต้นฉบับของ Hilpisch

มีผลจากการรัน Python ทุกเซลล์ตามลำดับจาก state ว่างแนบไว้แล้ว ไม่มีการเชื่อมเครือข่าย กด Restart Kernel แล้ว Run All เพื่อทำซ้ำใน Jupyter ได้